In [1]:
# ============================================================
# 1. IMPORTS, DATA LOADING & PREPARATION
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../data/processed/cleaned_stock_data.csv")
OUTPUT_PATH = Path("../data/processed/feature_engineered_data.csv")

# Load
stocks_df = pd.read_csv(DATA_PATH)

# Date conversion
stocks_df["Date"] = pd.to_datetime(stocks_df["Date"])

# Very important for lag/rolling calculations
stocks_df = (
    stocks_df
    .sort_values(["Symbol", "Date"])
    .reset_index(drop=True)
)

print("Shape:", stocks_df.shape)
print("Stocks:", stocks_df["Symbol"].nunique())
print(
    "Date Range:",
    stocks_df["Date"].min(),
    "→",
    stocks_df["Date"].max()
)

stocks_df.head()

Shape: (235192, 18)
Stocks: 49
Date Range: 2000-01-03 00:00:00 → 2021-04-30 00:00:00


,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Trades,Deliverable Volume,%Deliverble,Company Name,Industry,ISIN Code
0,2007-11-27,ADANIPORTS,EQ,440.00,770.00,1050.00,770.0,959.0,962.90,984.72,27294366,2.687719e+15,NaN,9859619.0,0.3612,Adani Ports and Special Economic Zone Ltd.,SERVICES,INE742F01042
1,2007-11-28,ADANIPORTS,EQ,962.90,984.00,990.00,874.0,885.0,893.90,941.38,4581338,4.312765e+14,NaN,1453278.0,0.3172,Adani Ports and Special Economic Zone Ltd.,SERVICES,INE742F01042
2,2007-11-29,ADANIPORTS,EQ,893.90,909.00,914.75,841.0,887.0,884.20,888.09,5124121,4.550658e+14,NaN,1069678.0,0.2088,Adani Ports and Special Economic Zone Ltd.,SERVICES,INE742F01042
3,2007-11-30,ADANIPORTS,EQ,884.20,890.00,958.00,890.0,929.0,921.55,929.17,4609762,4.283257e+14,NaN,1260913.0,0.2735,Adani Ports and Special Economic Zone Ltd.,SERVICES,INE742F01042
4,2007-12-03,ADANIPORTS,EQ,921.55,939.75,995.00,922.0,980.0,969.30,965.65,2977470,2.875200e+14,NaN,816123.0,0.2741,Adani Ports and Special Economic Zone Ltd.,SERVICES,INE742F01042


In [2]:
# ============================================================
# 2. RETURN, LAG, TREND & MOMENTUM FEATURES
# ============================================================

# -------------------------
# Return Features
# -------------------------

stocks_df["Daily_Return"] = (
    stocks_df.groupby("Symbol")["Close"]
    .pct_change()
)

stocks_df["Log_Return"] = (
    stocks_df.groupby("Symbol")["Close"]
    .transform(lambda x: np.log(x / x.shift(1)))
)

stocks_df["Intraday_Return"] = (
    (stocks_df["Close"] - stocks_df["Open"])
    / stocks_df["Open"]
)

stocks_df["High_Low_Range"] = (
    (stocks_df["High"] - stocks_df["Low"])
    / stocks_df["Open"]
)


# -------------------------
# Lag Features
# -------------------------

for lag in [1, 2, 3, 5, 10]:

    stocks_df[f"Return_Lag_{lag}"] = (
        stocks_df.groupby("Symbol")["Daily_Return"]
        .shift(lag)
    )


# -------------------------
# SMA + Price/SMA
# -------------------------

for window in [5, 10, 20, 50, 100, 200]:

    stocks_df[f"SMA_{window}"] = (
        stocks_df.groupby("Symbol")["Close"]
        .transform(
            lambda x: x.rolling(window).mean()
        )
    )

    stocks_df[f"Price_SMA_{window}_Ratio"] = (
        stocks_df["Close"]
        / stocks_df[f"SMA_{window}"]
    )


# -------------------------
# EMA
# -------------------------

stocks_df["EMA_12"] = (
    stocks_df.groupby("Symbol")["Close"]
    .transform(
        lambda x: x.ewm(
            span=12,
            adjust=False
        ).mean()
    )
)

stocks_df["EMA_26"] = (
    stocks_df.groupby("Symbol")["Close"]
    .transform(
        lambda x: x.ewm(
            span=26,
            adjust=False
        ).mean()
    )
)


# -------------------------
# Momentum
# -------------------------

for period in [5, 10, 20]:

    stocks_df[f"Momentum_{period}"] = (
        stocks_df.groupby("Symbol")["Close"]
        .pct_change(period)
    )


# -------------------------
# RSI
# -------------------------

def calculate_rsi(series, window=14):

    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()

    rs = avg_gain / avg_loss

    return 100 - (100 / (1 + rs))


stocks_df["RSI_14"] = (
    stocks_df.groupby("Symbol")["Close"]
    .transform(calculate_rsi)
)


# -------------------------
# MACD
# -------------------------

stocks_df["MACD"] = (
    stocks_df["EMA_12"]
    - stocks_df["EMA_26"]
)

stocks_df["MACD_Signal"] = (
    stocks_df.groupby("Symbol")["MACD"]
    .transform(
        lambda x: x.ewm(
            span=9,
            adjust=False
        ).mean()
    )
)

stocks_df["MACD_Histogram"] = (
    stocks_df["MACD"]
    - stocks_df["MACD_Signal"]
)


print("Return, trend and momentum features created.")

Return, trend and momentum features created.


In [3]:
# ============================================================
# 3. RISK, VOLUME & PRICE-ACTION FEATURES
# ============================================================

# -------------------------
# Bollinger Bands
# -------------------------

stocks_df["BB_Middle"] = (
    stocks_df.groupby("Symbol")["Close"]
    .transform(
        lambda x: x.rolling(20).mean()
    )
)

bb_std = (
    stocks_df.groupby("Symbol")["Close"]
    .transform(
        lambda x: x.rolling(20).std()
    )
)

stocks_df["BB_Upper"] = (
    stocks_df["BB_Middle"]
    + 2 * bb_std
)

stocks_df["BB_Lower"] = (
    stocks_df["BB_Middle"]
    - 2 * bb_std
)

stocks_df["BB_Position"] = (
    (stocks_df["Close"] - stocks_df["BB_Lower"])
    /
    (stocks_df["BB_Upper"] - stocks_df["BB_Lower"])
)


# -------------------------
# Rolling Volatility
# -------------------------

for window in [5, 10, 20, 30]:

    stocks_df[f"Volatility_{window}"] = (
        stocks_df.groupby("Symbol")["Daily_Return"]
        .transform(
            lambda x: x.rolling(window).std()
        )
    )


# -------------------------
# Volume Features
# -------------------------

stocks_df["Volume_Change"] = (
    stocks_df.groupby("Symbol")["Volume"]
    .pct_change()
)

stocks_df["Volume_MA_20"] = (
    stocks_df.groupby("Symbol")["Volume"]
    .transform(
        lambda x: x.rolling(20).mean()
    )
)

stocks_df["Relative_Volume"] = (
    stocks_df["Volume"]
    / stocks_df["Volume_MA_20"]
)


# -------------------------
# Opening Gap
# -------------------------

previous_close = (
    stocks_df.groupby("Symbol")["Close"]
    .shift(1)
)

stocks_df["Gap_Return"] = (
    stocks_df["Open"]
    / previous_close
    - 1
)


# -------------------------
# Close Position
# -------------------------

daily_range = (
    stocks_df["High"]
    - stocks_df["Low"]
)

stocks_df["Close_Position"] = np.where(
    daily_range != 0,
    (stocks_df["Close"] - stocks_df["Low"])
    / daily_range,
    np.nan
)


# -------------------------
# Rolling Mean Returns
# -------------------------

for window in [5, 10, 20]:

    stocks_df[f"Mean_Return_{window}"] = (
        stocks_df.groupby("Symbol")["Daily_Return"]
        .transform(
            lambda x: x.rolling(window).mean()
        )
    )


# -------------------------
# Return Distribution
# -------------------------

stocks_df["Return_Skew_20"] = (
    stocks_df.groupby("Symbol")["Daily_Return"]
    .transform(
        lambda x: x.rolling(20).skew()
    )
)

stocks_df["Return_Min_20"] = (
    stocks_df.groupby("Symbol")["Daily_Return"]
    .transform(
        lambda x: x.rolling(20).min()
    )
)

stocks_df["Return_Max_20"] = (
    stocks_df.groupby("Symbol")["Daily_Return"]
    .transform(
        lambda x: x.rolling(20).max()
    )
)


# -------------------------
# ATR
# -------------------------

tr1 = stocks_df["High"] - stocks_df["Low"]

tr2 = (
    stocks_df["High"]
    - previous_close
).abs()

tr3 = (
    stocks_df["Low"]
    - previous_close
).abs()

stocks_df["True_Range"] = np.maximum.reduce([
    tr1,
    tr2,
    tr3
])

stocks_df["ATR_14"] = (
    stocks_df.groupby("Symbol")["True_Range"]
    .transform(
        lambda x: x.rolling(14).mean()
    )
)

stocks_df["ATR_14_Pct"] = (
    stocks_df["ATR_14"]
    / stocks_df["Close"]
)


# -------------------------
# 52-Week Features
# -------------------------

stocks_df["High_252"] = (
    stocks_df.groupby("Symbol")["High"]
    .transform(
        lambda x: x.rolling(252).max()
    )
)

stocks_df["Low_252"] = (
    stocks_df.groupby("Symbol")["Low"]
    .transform(
        lambda x: x.rolling(252).min()
    )
)

stocks_df["Distance_52W_High"] = (
    stocks_df["Close"]
    / stocks_df["High_252"]
    - 1
)

stocks_df["Position_52W"] = (
    (stocks_df["Close"] - stocks_df["Low_252"])
    /
    (stocks_df["High_252"] - stocks_df["Low_252"])
)


print("Risk, volume and price-action features created.")

Risk, volume and price-action features created.


In [4]:
# ============================================================
# 4. MARKET, SECTOR & CALENDAR FEATURES
# ============================================================

# -------------------------
# Leave-One-Out Market Return
# -------------------------

market_stats = (
    stocks_df.groupby("Date")["Daily_Return"]
    .agg(["sum", "count"])
    .rename(
        columns={
            "sum": "Market_Return_Sum",
            "count": "Market_Stock_Count"
        }
    )
)

stocks_df = stocks_df.merge(
    market_stats,
    left_on="Date",
    right_index=True,
    how="left"
)

stocks_df["Market_Return_LOO"] = np.where(
    stocks_df["Market_Stock_Count"] > 1,

    (
        stocks_df["Market_Return_Sum"]
        - stocks_df["Daily_Return"]
    )
    /
    (
        stocks_df["Market_Stock_Count"]
        - 1
    ),

    np.nan
)

stocks_df["Relative_Market_Return_LOO"] = (
    stocks_df["Daily_Return"]
    - stocks_df["Market_Return_LOO"]
)


# -------------------------
# Leave-One-Out Sector Return
# -------------------------

sector_stats = (
    stocks_df
    .groupby(
        ["Date", "Industry"]
    )["Daily_Return"]
    .agg(["sum", "count"])
    .reset_index()
    .rename(
        columns={
            "sum": "Sector_Return_Sum",
            "count": "Sector_Stock_Count"
        }
    )
)

stocks_df = stocks_df.merge(
    sector_stats,
    on=["Date", "Industry"],
    how="left"
)

stocks_df["Sector_Return_LOO"] = np.where(
    stocks_df["Sector_Stock_Count"] > 1,

    (
        stocks_df["Sector_Return_Sum"]
        - stocks_df["Daily_Return"]
    )
    /
    (
        stocks_df["Sector_Stock_Count"]
        - 1
    ),

    np.nan
)

stocks_df["Relative_Sector_Return_LOO"] = (
    stocks_df["Daily_Return"]
    - stocks_df["Sector_Return_LOO"]
)


# -------------------------
# Market Breadth
# -------------------------

stocks_df["Stock_Up"] = (
    stocks_df["Daily_Return"] > 0
).astype(int)

breadth_stats = (
    stocks_df.groupby("Date")["Stock_Up"]
    .agg(["sum", "count"])
    .rename(
        columns={
            "sum": "Up_Count",
            "count": "Breadth_Count"
        }
    )
)

stocks_df = stocks_df.merge(
    breadth_stats,
    left_on="Date",
    right_index=True,
    how="left"
)

# Leave current stock out
stocks_df["Market_Breadth_LOO"] = np.where(
    stocks_df["Breadth_Count"] > 1,

    (
        stocks_df["Up_Count"]
        - stocks_df["Stock_Up"]
    )
    /
    (
        stocks_df["Breadth_Count"]
        - 1
    ),

    np.nan
)


# -------------------------
# Calendar Features
# -------------------------

stocks_df["Year"] = stocks_df["Date"].dt.year
stocks_df["Month"] = stocks_df["Date"].dt.month
stocks_df["DayOfWeek"] = stocks_df["Date"].dt.dayofweek
stocks_df["Quarter"] = stocks_df["Date"].dt.quarter


print("Market, sector and calendar features created.")

Market, sector and calendar features created.


In [5]:
# ============================================================
# 5. TARGET CREATION & DATA VALIDATION
# ============================================================

# -------------------------
# Prediction Targets
# -------------------------

for horizon in [1, 3, 5, 10]:

    future_return = (
        stocks_df.groupby("Symbol")["Close"]
        .shift(-horizon)
        / stocks_df["Close"]
        - 1
    )

    stocks_df[
        f"Future_Return_{horizon}D"
    ] = future_return

    stocks_df[
        f"Target_{horizon}D"
    ] = np.where(
        future_return > 0,
        1,
        0
    )

    # Prevent unavailable future observations
    # from becoming false DOWN labels
    stocks_df.loc[
        future_return.isna(),
        f"Target_{horizon}D"
    ] = np.nan


# Main target alias
stocks_df["Target_Direction"] = (
    stocks_df["Target_1D"]
)

stocks_df["Future_Return_1D"] = (
    stocks_df["Future_Return_1D"]
)


# -------------------------
# Replace Infinite Values
# -------------------------

numeric_cols = stocks_df.select_dtypes(
    include=np.number
).columns

stocks_df[numeric_cols] = (
    stocks_df[numeric_cols]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)


# -------------------------
# Validation
# -------------------------

print("Final Shape:", stocks_df.shape)

print(
    "\nNumber of numeric columns:",
    len(numeric_cols)
)

print("\nMain Target Distribution:")

print(
    stocks_df["Target_Direction"]
    .value_counts(
        normalize=True,
        dropna=True
    )
    .mul(100)
    .round(2)
)

missing = (
    stocks_df.isnull()
    .sum()
    .sort_values(ascending=False)
)

print("\nTop missing-value columns:")

print(
    missing[
        missing > 0
    ].head(30)
)

Final Shape: (235192, 99)

Number of numeric columns: 93

Main Target Distribution:
Target_Direction
1.0    50.35
0.0    49.65
Name: proportion, dtype: float64

Top missing-value columns:
Trades                        114848
Sector_Return_LOO              22238
Relative_Sector_Return_LOO     22238
Deliverable Volume             16077
%Deliverble                    16077
High_252                       12299
Position_52W                   12299
Distance_52W_High              12299
Low_252                        12299
SMA_200                         9751
Price_SMA_200_Ratio             9751
SMA_100                         4851
Price_SMA_100_Ratio             4851
SMA_50                          2401
Price_SMA_50_Ratio              2401
Volatility_30                   1470
Mean_Return_20                   980
Momentum_20                      980
Return_Max_20                    980
Return_Min_20                    980
Volatility_20                    980
Return_Skew_20                   98

In [6]:
# ============================================================
# 6. FEATURE GROUPS & SAVE DATASET
# ============================================================

# Original engineered features
base_features = [

    "Daily_Return",
    "Log_Return",
    "Intraday_Return",
    "High_Low_Range",

    "Return_Lag_1",
    "Return_Lag_2",
    "Return_Lag_3",
    "Return_Lag_5",
    "Return_Lag_10",

    "Price_SMA_5_Ratio",
    "Price_SMA_10_Ratio",
    "Price_SMA_20_Ratio",
    "Price_SMA_50_Ratio",
    "Price_SMA_100_Ratio",
    "Price_SMA_200_Ratio",

    "Momentum_5",
    "Momentum_10",
    "Momentum_20",

    "RSI_14",

    "MACD",
    "MACD_Signal",
    "MACD_Histogram",

    "BB_Position",

    "Volatility_5",
    "Volatility_10",
    "Volatility_20",
    "Volatility_30",

    "Volume_Change",
    "Relative_Volume",

    "Month",
    "DayOfWeek",
    "Quarter"
]


advanced_features = [

    "Gap_Return",
    "Close_Position",

    "Mean_Return_5",
    "Mean_Return_10",
    "Mean_Return_20",

    "Return_Skew_20",
    "Return_Min_20",
    "Return_Max_20",

    "ATR_14_Pct",

    "Distance_52W_High",
    "Position_52W",

    "Market_Return_LOO",
    "Relative_Market_Return_LOO",

    "Sector_Return_LOO",
    "Relative_Sector_Return_LOO",

    "Market_Breadth_LOO"
]


features_v2 = (
    base_features
    + advanced_features
)


print("Base Features:", len(base_features))
print("Advanced Features:", len(advanced_features))
print("Total Candidate Features:", len(features_v2))


# Save
stocks_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    "\nFeature-engineered dataset saved to:",
    OUTPUT_PATH
)

Base Features: 32
Advanced Features: 16
Total Candidate Features: 48

Feature-engineered dataset saved to: ..\data\processed\feature_engineered_data.csv
